# ROI representation (NeoSTAR): reproducible pipeline

This notebook reproduces the paper **"NeoSTAR: A Stable, Auditable Dynamic ROI Representation for Neonatal Video"**
step by step: its tables, figures and the full manuscript. Run the cells from top to bottom.

| Level | What it reproduces | Needs | Time |
|---|---|---|---|
| **A** | Every table, figure, statistic and the manuscript, from the released result tables | Python only (CPU) | minutes |
| **C** | The neonatal results (stability, validity tests, consistency, cost) from the 34 videos | the 34 neonatal videos, a GPU | about 2 days |
| **B** | The heart-rate robustness study on UBFC-rPPG with controlled perturbations | UBFC-rPPG, a GPU, Level C's base run | about 1-2 days |

Level A needs no data. Levels B and C re-run the experiments from raw video and then compare every result table with
the released one (`tools/verify_results.py`). The neonatal videos are third-party web videos and are not distributed;
UBFC-rPPG must be requested from its authors; model weights are downloaded in Step 3.

**Folder map.** `module1_v2/` NeoSTAR code, configuration and manuscript builder · `src/`, `scripts/`, `configs/` the
detection-segmentation-pose pipeline it builds on · `HR RR using module1 v2/` the heart-rate chain of the robustness
study · `dataaugment/` motion, lighting and occlusion generators · `module1_v2_d8/`, `module1_v2_d9/`,
`occlusion_position_test/`, `face_track_baseline/` the swaddling, occlusion and face-box studies (with their
pre-registrations) · `released/` untouched copies of all result tables · `paper/` the submitted manuscript package ·
`tools/` setup and verification scripts · `config/paths.yaml` dataset locations.

## 0. Environment (once, in a terminal)

```bash
conda create -n neostar python=3.12 -y && conda activate neostar
pip install -r environment/requirements-paper.txt      # Level A only
# Levels B and C additionally:
pip install -r environment/requirements-full.txt
bash tools/install_external.sh                         # SAM 2 and Sapiens2 at the commits used for the paper
python -m ipykernel install --user --name neostar      # then select the 'neostar' kernel for this notebook
```
The results were produced with Python 3.12.15, PyTorch 2.11.0 (CUDA 12.8), OpenCV 4.14.0 and NumPy 2.5.4 on an
NVIDIA A100 (80 GB); see `module1_v2/reports/final34_paper/environment_versions.json`.

In [ ]:
import os, subprocess, sys, time
from pathlib import Path
ROOT = Path.cwd()
assert (ROOT/'tools/setup_paths.py').is_file(), 'Open this notebook from the ROI_representation folder'
sys.path.insert(0, str(ROOT/'tools'))
LOGS = ROOT/'logs'; LOGS.mkdir(exist_ok=True)
ENV = dict(os.environ, PY=sys.executable, PYTHONPATH=os.pathsep.join([str(ROOT/'src'), str(ROOT/'module1_v2/src')]))

def run(title, cmd, done=None, shell=False):
    """Run one step from the repository root; skipped when `done` (a path) already exists. Output goes to logs/."""
    if done is not None and Path(ROOT/done).exists():
        print(f'[skip] {title}: {done} exists'); return
    log = LOGS/(''.join(ch if ch.isalnum() else '_' for ch in title)[:60] + '.log')
    if not shell and str(cmd[0]).endswith('.py'):
        cmd = [sys.executable] + [str(c) for c in cmd]
    print(f'[run ] {title}\n       {cmd if shell else " ".join(map(str, cmd))}\n       log: {log.relative_to(ROOT)}')
    t0 = time.time()
    with open(log, 'w') as fh:
        p = subprocess.run(cmd, cwd=ROOT, env=ENV, shell=shell, stdout=fh, stderr=subprocess.STDOUT,
                           executable='/bin/bash' if shell else None)
    tail = log.read_text().strip().splitlines()[-8:]
    print('       ' + '\n       '.join(tail))
    print(f'[{"done" if p.returncode == 0 else "FAILED"}] {title} ({(time.time() - t0) / 60:.1f} min)')
    if p.returncode:
        raise RuntimeError(f'{title} failed, see {log}')
print('Repository:', ROOT, '| Python', sys.version.split()[0])

## Step 1. Dataset paths (the only cell you edit)

Give the folders on your machine. Leave a path empty if you do not have that data; the steps that need it are skipped.
The values are saved to `config/paths.yaml`, so you can also edit that file instead.

In [ ]:
NEONATAL_VIDEOS    = ''                          # folder with the 34 neonatal .mp4 files (Level C)
UBFC_RPPG          = ''                          # UBFC-rPPG DATASET_2 folder containing subject1/, subject3/, subject4/ (Level B)
DATAAUGMENT_VIDEOS = 'work/dataaugment_videos'   # DataAugment videos: existing folder with data/, data_motion_fast/,
                                                 #   data_motion_fast_xl/, or an empty folder that Step B3 fills
DERIVED_VIDEOS     = 'work/derived_videos'       # where the other generated videos are written (Level B)
MODELS             = 'models'                    # model weights (Step 3 downloads them here)

RUN_LEVEL_C = False      # neonatal pipeline from raw video (GPU, about 2 days)
RUN_LEVEL_B = False      # UBFC robustness study from raw video (GPU, about 1-2 days; needs Level C's base run)
WORKERS     = 2          # CPU worker processes for the evaluation steps

from common import write_paths
write_paths(neonatal_videos=NEONATAL_VIDEOS, ubfc_rppg=UBFC_RPPG, dataaugment_videos=DATAAUGMENT_VIDEOS,
            derived_videos=DERIVED_VIDEOS, models=MODELS)
print(Path('config/paths.yaml').read_text())

## Step 2. Connect the datasets

The code expects fixed locations (`data/final34/dataset`, `data/ubfc/subjectN`, `dataaugment/data`, `models`, ...).
`tools/setup_paths.py` makes each of them a link to your folder, writes your local video paths into the neonatal
manifest and checks the files. With `--hash` it also verifies the SHA-256 of every neonatal video against
`runs/final34_paper/manifest.csv` (the video-level split depends on the exact files).

In [ ]:
run('link and check datasets', ['tools/setup_paths.py'] + (['--hash'] if RUN_LEVEL_C else []))
print(open(LOGS/'link_and_check_datasets.log').read())

## Step 3. Model weights and external code (Levels B and C)

Four pretrained models are used, none trained or fine-tuned here: YOLO11s and RT-DETR-L (infant detection), SAM 2.1
Hiera-T (infant mask) and Sapiens2-0.4B (pose). Their SHA-256 hashes are checked against the files used for the paper.

In [ ]:
if RUN_LEVEL_B or RUN_LEVEL_C:
    run('install SAM 2 and Sapiens2 (pinned commits)', 'bash tools/install_external.sh', done='external/sapiens2/pyproject.toml', shell=True)
    run('download and check model weights', ['tools/download_models.py'])
else:
    print('Level A only: no models needed')

---
# Level A. The paper from the released results

### A1. Rebuild the manuscript
Every number in the text is computed from the result tables in this repository (nothing is typed by hand); the build
fails if an edit does not apply, if a citation or table number is inconsistent, or if a word limit is exceeded.
Output: `module1_v2/reports/final34_paper/submission_results_in_engineering/` (manuscript, highlights, cover letter,
supplement, figures, checklist). To target another journal, add a profile in `module1_v2/manuscript/journals/`.

In [ ]:
run('A1 rebuild manuscript', ['module1_v2/manuscript/build_submission.py', '--name', 'final34_paper',
                               '--v1-report', 'reports/paper_final34_paper', '--journal', 'results_in_engineering'])

### A2. Check against the released manuscript

In [ ]:
run('A2 compare with released manuscript', ['tools/verify_results.py', 'paper'])

### A3. Main results

In [ ]:
import pandas as pd
from IPython.display import display, Image
pd.set_option('display.width', 200, 'display.max_columns', 30)
P1 = ROOT/'module1_v2/reports/final34_paper/phase1'
print('Temporal stability and pulse quality on the 34 neonatal videos (mean over videos):')
pv = pd.read_csv(P1/'per_video_metrics.csv')
cols = ['switches_per_sec', 'temporal_iou', 'retention', 'pulse_snr_db', 'informative_window_percent']
display(pv[pv.config.isin(['B0', 'Full', 'D3', 'D3L', 'D5'])].groupby('config')[cols].mean().round(3)
        .rename(index={'Full': 'PF-ROI (Full)', 'D5': 'NeoSTAR (D5)', 'B0': 'fixed box (B0)'}))
sys.path.insert(0, str(ROOT/'module1_v2/manuscript'))
import robustness_sub as RS
T = RS.load()['T']
print('Heart-rate error under controlled perturbations (UBFC-rPPG, fingertip-PPG reference; MAE in bpm):')
display(T[['condition', 'level', 'subjects', 'windows', 'mae_fixed', 'mae_v1', 'mae_FACE', 'mae_D5', 'diff', 'lo', 'hi',
           'verdict', 'design']].round(2))

In [ ]:
FIG = ROOT/'module1_v2/reports/final34_paper/submission_results_in_engineering/figures'
for f in sorted(FIG.glob('Figure_*.png'), key=lambda p: int(p.stem.split('_')[1])):
    print(f.stem); display(Image(filename=str(f), width=900))

### A4. Unit tests of the NeoSTAR code (CPU)

In [ ]:
run('A4 unit tests', [sys.executable, '-m', 'pytest', '-q', 'module1_v2/tests', 'module1_v2_d8/tests'])

---
# Level C. Neonatal pipeline from raw video (34 videos)

Needs `NEONATAL_VIDEOS` and `RUN_LEVEL_C = True`. The video-level split (21 training, 7 validation, 6 test; seeded
shuffle, seed 42) is fixed in `runs/final34_paper/manifest.csv` and was checked by hash in Step 2. Each step is skipped
when its output exists, so the notebook can be re-run to resume. Run times are those of the original run.

| Step | What | Time |
|---|---|---|
| C1 | base pipeline: detection, tracking, recovery, SAM 2, Sapiens2 pose, 17 candidate ROIs, condition terms (GPU) | ~21 h |
| C2 | PF-ROI evaluation (sets the informative-SNR threshold τ used by every later step) | ~1.5 h |
| C3 | NeoSTAR Phase 1: condition terms, stabilizer tuning on training videos, evaluation | ~1.5 h |
| C4 | Phase 2 validity tests (199 surrogates, background control), fusion test, consistency, Phase 5 cost | ~10 h |
| C5 | Phase 1 report figures, then compare all tables with the released ones | minutes |

In [ ]:
if RUN_LEVEL_C:
    run('C1 base pipeline (GPU)', ['scripts/run_paper_pilot.py', '--name', 'final34_paper'],
        done='runs/final34_paper/outputs/confidence_sam2/roi_confidence.csv')
    run('C2 PF-ROI evaluation', ['scripts/run_paper_evaluation.py', '--run', 'runs/final34_paper', '--output-dir',
        'reports/paper_final34_paper', '--gpu-log', 'runs/final34_paper/gpu_monitor.csv', '--workers', str(WORKERS)],
        done='reports/paper_final34_paper/per_video_metrics.csv')
else:
    print('Level C skipped (set RUN_LEVEL_C = True in Step 1)')

In [ ]:
if RUN_LEVEL_C:
    V = ['--run', 'runs/final34_paper', '--v1-report', 'reports/paper_final34_paper', '--workers', str(WORKERS)]
    run('C3 NeoSTAR phase 1 (features, tuning, evaluation)', ['module1_v2/scripts/run_phase1.py'] + V)
    run('C4a phase 2 validity tests', ['module1_v2/scripts/run_phase2.py'] + V)
    run('C4b fusion-weight test', ['module1_v2/scripts/run_fusion_test.py'] + V)
    run('C4c homography features', ['module1_v2/scripts/run_v2_features.py', '--run', 'runs/final34_paper',
        '--global-motion', 'homography', '--workers', str(WORKERS)],
        done='module1_v2/runs/final34_paper/features/roi_features_homography.csv')
    run('C4d consistency E1-E4', ['module1_v2/scripts/run_consistency.py', '--run', 'runs/final34_paper', '--workers', str(WORKERS)])
    run('C4e consistency C1-C4', ['module1_v2/scripts/run_consistency_checks.py', '--run', 'runs/final34_paper', '--workers', str(WORKERS)])
    run('C4f phase 5 cost', ['module1_v2/scripts/run_phase5.py'] + V + ['--configs', 'D0,D3,D5'])
    run('C5a phase 1 figures', ['module1_v2/scripts/build_phase1_report.py', '--name', 'final34_paper', '--headline', 'D5'])
    run('C5b compare neonatal results with released', ['tools/verify_results.py', 'neonatal'])

Two released folders are historical and are not regenerated: `phase1_prefix/`, `consistency_prefix/` (results before
the two-point-box fix, quoted in Section 5.1) and `phase2_s19/` (the earlier 19-surrogate run, used for the Monte-Carlo
check in Section 5.5). The headline configuration (D5) was fixed in `headline.json` by the validation rule of Section
4.4 before the test split was examined.

---
# Level B. Heart-rate robustness on UBFC-rPPG

Needs `UBFC_RPPG`, `RUN_LEVEL_B = True` and the GPU models. Nothing is tuned here: NeoSTAR uses the stabilizer settings
tuned on the neonatal training videos (`module1_v2/runs/final34_paper/tuning_log.json`). The swaddling studies also
re-check NeoSTAR on the neonatal videos (no-harm check), which needs Level C's base run.

| Step | What | Time |
|---|---|---|
| B1 | lossless copies of UBFC subjects 1, 3, 4 and the fingertip-PPG reference HR | ~10 min |
| B2 | base pipeline on the original videos | ~1.6 h |
| B3 | DataAugment videos: slow motion, occlusion, lighting; fast motion incl. extra-large (seed 20261114) | ~20 min |
| B4 | fixed box, PF-ROI and NeoSTAR on the same HR chain; report | ~9 h |
| B5 | swaddling videos and their evaluation (D8 chain; also writes the per-condition metrics table) | ~6 h |
| B6 | designed hand-over-the-face-centre video (supplement S4) | ~30 min |
| B7 | pre-registered swaddling-only and swaddling + fast motion tests | ~12 h |
| B8 | tracked face-box baseline on all conditions | ~2.5 h |
| B9 | compare all tables with the released ones | minutes |

In [ ]:
if RUN_LEVEL_B:
    run('B1 UBFC copies and reference HR', ['tools/prepare_ubfc.py'],
        done='HR RR using module1 v2/runs/ubfc_flat_38d8f553/ubfc_reference/reference_hr_oximeter.csv')
    run('B2a base pipeline on original UBFC (GPU)', ['scripts/run_new_video.py', 'data/ubfc_flat', '--branch', 'sam2',
        '--name', 'ubfc_flat_38d8f553'], done='runs/ubfc_flat_38d8f553/outputs/confidence_sam2/roi_confidence.csv')
    run('B2b NeoSTAR features on original UBFC', ['module1_v2/scripts/run_v2_features.py', '--run',
        'runs/ubfc_flat_38d8f553', '--workers', str(WORKERS)],
        done='module1_v2/runs/ubfc_flat_38d8f553/features/roi_features_median.csv')
else:
    print('Level B skipped (set RUN_LEVEL_B = True in Step 1)')

In [ ]:
if RUN_LEVEL_B:
    run('B3a DataAugment: motion, occlusion, lighting', ['dataaugment/main.py'], done='dataaugment/data/manifest.csv')
    run('B3b DataAugment: fast motion', ['dataaugment/main.py', '--config', 'dataaugment/config/motion_fast_config.yaml'],
        done='dataaugment/data_motion_fast/manifest.csv')
    run('B3c DataAugment: fast motion extra-large', ['dataaugment/main.py', '--config',
        'dataaugment/config/motion_fast_xl_config.yaml'], done='dataaugment/data_motion_fast_xl/manifest.csv')
    run('B3d collect videos for the HR evaluation', ['tools/link_augmented_videos.py'])
    run('B4 HR evaluation (fixed box, PF-ROI, NeoSTAR)', 'bash dataaugment/hr_eval/run_all.sh strict', shell=True)

In [ ]:
if RUN_LEVEL_B:
    run('B5a swaddling videos', ['module1_v2_d8/scripts/make_swaddle_videos.py'],
        done='module1_v2_d8/data_swaddle/subject1_swaddle_large.mkv')
    run('B5b swaddling evaluation (D8 chain)', 'bash module1_v2_d8/run_d8_eval.sh', shell=True)
    run('B6a hand-over-face video', ['occlusion_position_test/demo_centre/make_demo_video.py'],
        done='occlusion_position_test/demo_centre/subject1_democentre.mkv')
    run('B6b hand-over-face evaluation', 'bash occlusion_position_test/demo_centre/run_demo.sh', shell=True)
    run('B7 pre-registered swaddling tests', 'bash module1_v2_d9/run_chain.sh', shell=True)
    run('B8 tracked face box', 'bash face_track_baseline/run_face_track.sh', shell=True)
    run('B9 compare UBFC results with released', ['tools/verify_results.py', 'ubfc'])

---
# Final. Rebuild the paper from your re-computed results

After Level B and/or C, the result tables in place are your own. Rebuilding the manuscript then gives a paper computed
entirely from your run; `verify_results.py paper` shows whether any number in the text changed.

In [ ]:
if RUN_LEVEL_B or RUN_LEVEL_C:
    run('rebuild manuscript from re-computed results', ['module1_v2/manuscript/build_submission.py', '--name',
        'final34_paper', '--v1-report', 'reports/paper_final34_paper'])
    try:
        run('compare with released manuscript', ['tools/verify_results.py', 'paper'])
    except RuntimeError:
        print('The text differs: see the first difference above (small numeric differences are expected after a GPU re-run).')